# Section 1 — Contract Destination Analysis

**Issue #14 / PR #28.** Compare four fixed destination rules for ZC, ZS, and ZW while keeping baseline roll-event dates, target weights, capacity, rounding, benchmark, and exact-contract P&L unchanged.

**Status:** Code integration staged; empirical outputs are not yet verified. This notebook does not fabricate returns or silently replace missing historical data.

## Design

The four rules are **next deferred**, **second deferred** (one-candidate fallback to next), **closest to 45 observed sessions**, and **closest to 75 observed sessions**. Destination selection occurs at baseline roll events, not every day. The strategy's actual outgoing contract determines its deferred candidates.

Use the Section 0 Phase B frozen handoff, a **$10 million** reference AUM, and shared `holdings_from_contract_path` and `portfolio_pnl`. Rank on realized daily Tracking RMSE (bp) over the same common date mask. The sample requires at least **100** common valid dates and **70%** survival. A difference strictly below **0.1 bp** is a tie; the agreed Part V tie order is required to resolve it.

## Reproduce

Run these commands in a **combined checkout** containing both PR #28 and Section 0 Phase B (PR #29), with `data/section0` available.

```bash
uv sync --locked
uv run --no-sync python -B -m unittest discover -s tests/unit -p 'test_selection*.py' -v
uv run --no-sync python -B scripts/run_person1.py --real
```

The runner refuses to export empirical results if a destination path is unimplemented, the frozen benchmark does not reproduce, or the coverage gate fails. Do not substitute synthetic results.

In [ ]:
from pathlib import Path
import csv

root = Path.cwd()
if not (root / "outputs/person1").exists() and (root.parent / "outputs/person1").exists():
    root = root.parent
outdir = root / "outputs/person1"
summary_file = outdir / "contract_destination_summary.csv"
daily_file = outdir / "contract_destination_daily.csv"
print("Empirical outputs present:", summary_file.is_file() and daily_file.is_file())
print("Output directory:", outdir)

In [ ]:
if summary_file.is_file():
    with summary_file.open(newline="") as f:
        summary = list(csv.DictReader(f))
    for row in summary:
        print(row)
else:
    print("Not yet run: no empirical summary CSV. Run the real-data command above.")

In [ ]:
if daily_file.is_file():
    with daily_file.open(newline="") as f:
        daily = list(csv.DictReader(f))
    print("Daily observations:", len(daily))
    print("Common valid dates:", sum(r["in_common_mask"].lower() == "true" for r in daily))
else:
    print("Not yet run: no empirical daily CSV.")

In [ ]:
from IPython.display import Image, display
for filename in ("tracking_rmse_by_rule.png", "maturity_exposure.png"):
    path = outdir / filename
    if path.is_file():
        display(Image(filename=str(path)))
    else:
        print(f"Not yet generated: {filename}")

## Interpretation and limitations (complete after real run)

Record the realized RMSE for each rule, the common valid-date count, survival, roll/fallback counts, and maturity exposure. Identify the preferred rule **only if** coverage passes and no unresolved tie remains. Explain any unimplemented contract paths or unavailable settlement observations instead of treating them as zero returns.

**No empirical conclusion is asserted before the frozen-data run succeeds.**